# 02. 실험 3개를 기록하고 데이터셋 리니지 남기기

책 4.4~4.6절. Decision Tree, Random Forest, XGBoost를 같은 데이터 분할로 학습합니다.
run마다 학습, 테스트, 피처 데이터셋을 SeaweedFS에 올리고 `log_input`으로 run에 연결합니다. 이 연결이 "이 모델은 어느 데이터로 만들었나"에 답하는 리니지입니다.
책과 달리 `random_state`를 고정해 같은 데이터와 파라미터로 다시 학습하면 같은 점수가 나오게 합니다.

In [ ]:
import pickle
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from income import load_income_data, upload_df

SEED = 42
BUCKET = "mlflow-datasets"
mlflow.set_experiment("income-classifier")

In [ ]:
# 책 4.4절과 같은 인코딩. column_list.pkl은 추론할 때 같은 one-hot 컬럼을 만드는 데 쓴다.
df = load_income_data()
feature_df = df.drop("Target", axis=1)
y = (df.Target.str.strip() == ">50K").astype(int).to_numpy()
X = pd.get_dummies(feature_df, drop_first=True, dtype=float)
col_list = X.columns.to_list()
with open("column_list.pkl", "wb") as f:
    pickle.dump(col_list, f)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=SEED)
X.shape, X_train.shape, X_test.shape

In [ ]:
def log_datasets():
    """학습, 테스트, 피처 데이터셋을 SeaweedFS에 올리고 run의 입력으로 기록한다.

    MLflow는 이름과 내용 해시(digest)가 같은 데이터셋을 experiment 안에 한 번만 저장한다.
    경로를 run ID가 아니라 digest로 정해, run이 달라도 같은 데이터는 같은 경로를 가리키게 한다.
    """
    train_df = X_train.assign(Target=y_train)
    test_df = X_test.assign(Target=y_test)
    for frame, name, context in [(train_df, "train", "training"), (test_df, "test", "testing"), (feature_df, "features", "reference")]:
        digest = mlflow.data.from_pandas(frame, name=name).digest
        uri = upload_df(frame, BUCKET, f"income-classifier/datasets/{name}-{digest}.csv")
        mlflow.log_input(mlflow.data.from_pandas(frame, source=uri, name=name), context=context)


def log_scores(p_train, p_test):
    scores = {
        "roc_auc_score_train": roc_auc_score(y_train, p_train),
        "roc_auc_score_test": roc_auc_score(y_test, p_test),
        "training_accuracy": accuracy_score(y_train, p_train > 0.5),
        "test_accuracy": accuracy_score(y_test, p_test > 0.5),
    }
    mlflow.log_metrics(scores)
    return scores

In [ ]:
results = {}
for run_name, model in [("decision-tree", DecisionTreeClassifier(random_state=SEED)),
                        ("random-forest", RandomForestClassifier(random_state=SEED, n_jobs=-1))]:
    with mlflow.start_run(run_name=run_name) as run:
        log_datasets()
        model.fit(X_train, y_train)
        results[run_name] = log_scores(model.predict_proba(X_train)[:, 1], model.predict_proba(X_test)[:, 1])
        mlflow.log_params(model.get_params())
        mlflow.log_artifact("column_list.pkl")
        # MLflow 3.17은 sklearn 모델을 skops로 저장하고 모르는 타입을 거절한다. 트리 타입만 신뢰 목록에 넣는다.
        info = mlflow.sklearn.log_model(model, name="income-classifier", input_example=X_test.head(3),
                                        skops_trusted_types=["sklearn.tree._tree.Tree"])
        # MLflow 3부터 모델은 run 아티팩트가 아니라 별도 LoggedModel(models:/m-...)로 저장된다. 등록할 때 쓰려고 run에 남긴다.
        mlflow.set_tag("model_uri", info.model_uri)
pd.DataFrame(results).T

In [ ]:
# XGBoost는 autolog가 파라미터와 학습 곡선을 기록한다. 모델은 다른 run과 같은 이름으로 직접 남긴다.
params = {"objective": "binary:logistic", "max_depth": 10, "learning_rate": 0.3, "seed": SEED}
with mlflow.start_run(run_name="xgboost") as run:
    # 데이터셋은 log_datasets()가 S3 경로로 남기므로 autolog의 데이터셋 기록은 끈다.
    mlflow.xgboost.autolog(log_models=False, log_datasets=False)
    log_datasets()
    dtrain, dtest = xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_test, label=y_test)
    booster = xgb.train(params, dtrain, num_boost_round=30, evals=[(dtest, "test")], verbose_eval=False)
    results["xgboost"] = log_scores(booster.predict(dtrain), booster.predict(dtest))
    mlflow.log_artifact("column_list.pkl")
    info = mlflow.xgboost.log_model(booster, name="income-classifier", input_example=X_test.head(3))
    mlflow.set_tag("model_uri", info.model_uri)
    mlflow.xgboost.autolog(disable=True)
pd.DataFrame(results).T

In [ ]:
# MLflow UI의 Experiments 화면과 같은 비교를 코드로 본다.
runs = mlflow.search_runs(experiment_names=["income-classifier"], filter_string="metrics.roc_auc_score_test > 0")
runs[["tags.mlflow.runName", "metrics.roc_auc_score_test", "metrics.test_accuracy", "run_id"]].sort_values("metrics.roc_auc_score_test", ascending=False)